# Kinetic-energy resonance figure

Interactive version of `plot_KineticEnergy_summary.py`.

Reads the summary produced by `batch_KineticEnergy.py`
(`KineticEnergy_summary.xlsx` or `.csv`) and draws two panels,
mean(Ek) and std(Ek) versus **f\* = flib/frot**:

- the main resonance sweep (most common dphi) as a connected curve,
- runs at other dphi as distinct square markers,
- repeat acquisitions of an identical (frot, flib, dphi) point (e.g. `_SS1`/`_SS2`)
  as open diamonds.

Set `SUMMARY_FILE` below, then run all cells.

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib qt

In [2]:
# Path to the summary produced by batch_KineticEnergy.py (.xlsx or .csv)
SUMMARY_FILE = ("/Users/jeromenoir/Documents/MyDocuments/LOCAL_PROJECT/"
                "TOPOGRAPHY_LIBRATION/CylinderExperimentsGMA/k6_TopBottom/"
                "KineticEnergy_summary.xlsx")

# Where to save the figure (set OUTPUT_FILE = None to skip saving)
OUTPUT_FILE = os.path.join(os.path.dirname(SUMMARY_FILE),
                           "KineticEnergy_vs_fstar.png")

In [3]:
def load_summary(path):
    """Read the summary table from .xlsx or .csv into a DataFrame."""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(path)
    if ext == ".csv":
        return pd.read_csv(path)
    raise ValueError("Unsupported summary format: %s" % ext)

In [4]:
def plot_summary(df, output_path=None):
    """Draw mean(Ek) and std(Ek) versus f*; optionally save to output_path."""
    dfv = df.dropna(subset=["fstar"]).copy()
    if dfv.empty:
        print("Nothing to plot: no valid f* values.")
        return None

    # Flag repeats: 2nd+ run sharing the same (frot, flib, dphi).
    dfv = dfv.sort_values(["frot_Hz", "flib_Hz", "dphi_deg", "run"])
    dfv["is_repeat"] = (dfv.groupby(["frot_Hz", "flib_Hz", "dphi_deg"])
                          .cumcount() > 0)
    dfv = dfv.sort_values("fstar")

    main_dphi = dfv.loc[~dfv["is_repeat"], "dphi_deg"].mode().iloc[0]
    is_main = (dfv["dphi_deg"] == main_dphi) & ~dfv["is_repeat"]

    panels = [("mean_Ekin", r"$\langle E_k \rangle$  (m$^2$/s$^2$)",
               "Mean kinetic energy"),
              ("std_Ekin", r"std $E_k$  (m$^2$/s$^2$)",
               "Std of kinetic energy")]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, (col, ylabel, title) in zip(axes, panels):
        main = dfv[is_main]
        ax.plot(main["fstar"], main[col], "-o", color="C0",
                label=r"$\delta\varphi=%g^\circ$ (sweep)" % main_dphi)
        for dphi, g in dfv[~is_main & ~dfv["is_repeat"]].groupby("dphi_deg"):
            ax.plot(g["fstar"], g[col], "s", ms=7,
                    label=r"$\delta\varphi=%g^\circ$" % dphi)
        rep = dfv[dfv["is_repeat"]]
        if not rep.empty:
            ax.plot(rep["fstar"], rep[col], "D", ms=8, mfc="none",
                    mec="k", mew=1.5, label="repeat")
        ax.set_yscale("log")
        ax.set_xlabel(r"$f^* = f_{\mathrm{lib}} / f_{\mathrm{rot}}$", fontsize=13)
        ax.set_ylabel(ylabel, fontsize=13)
        ax.set_title(title, fontsize=14)
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=9)

    fig.tight_layout()
    if output_path:
        fig.savefig(output_path, dpi=200, bbox_inches="tight")
        print("Figure written to:\n  %s" % output_path)
    return fig

In [5]:
df = load_summary(SUMMARY_FILE)
print("Loaded %d runs from %s" % (len(df), SUMMARY_FILE))
df.head()

Loaded 29 runs from /Users/jeromenoir/Documents/MyDocuments/LOCAL_PROJECT/TOPOGRAPHY_LIBRATION/CylinderExperimentsGMA/k6_TopBottom/KineticEnergy_summary.xlsx


,run,run idx,processed,frot_Hz,flib_Hz,dphi_deg,fstar,calibrated,dt_vel_s,fps_Hz,nframes,mean_Ekin,std_Ekin,npz
0,frot050_flib0400_dphi2deg_SS1,1,True,0.5,0.400,2.0,0.80,1.0,0.02,50.0,1500.0,0.000009,0.000002,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...
1,frot050_flib0400_dphi2deg_SS2,2,True,0.5,0.400,2.0,0.80,1.0,0.02,50.0,1500.0,0.000008,0.000001,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...
2,frot050_flib0405_dphi2deg_SS1,1,True,0.5,0.405,2.0,0.81,1.0,0.02,50.0,1500.0,0.000010,0.000002,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...
3,frot050_flib0410_dphi2deg_SS1,1,True,0.5,0.410,2.0,0.82,1.0,0.02,50.0,1500.0,0.000010,0.000001,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...
4,frot050_flib0415_dphi2deg_SS1,1,True,0.5,0.415,2.0,0.83,1.0,0.02,50.0,1500.0,0.000010,0.000001,/Users/jeromenoir/Documents/MyDocuments/LOCAL_...


## Filtering (optional)

Keep only runs matching a given **frot**, **flib**, **fstar**, and/or **dphi**.
Set a value (or a list of values) to filter on that quantity; leave it as
`None` to ignore it. Filters combine with AND, and float columns are matched
with a small tolerance.

Examples:
- `SELECT_DPHI = 2.0` → only the 2° sweep.
- `SELECT_FSTAR = 3.0` → only runs at f\* = 3 (the dphi sweep).
- `SELECT_FLIB = [0.40, 0.44]` → only those two libration frequencies.
- `SELECT_FROT = 0.5` → only runs at frot = 0.5 Hz.

In [8]:
SELECT_FROT  = None    # Hz, e.g. 0.5
SELECT_FLIB  = None    # Hz, e.g. 0.44  or  [0.40, 0.44]
SELECT_FSTAR = None    # e.g. 3.0
SELECT_DPHI  = 2.0    # deg, e.g. 2.0


def _match(series, sel, tol=1e-6):
    """Boolean mask: True where series matches sel (scalar or list of values),
    or all True when sel is None."""
    if sel is None:
        return pd.Series(True, index=series.index)
    vals = sel if isinstance(sel, (list, tuple, set)) else [sel]
    mask = pd.Series(False, index=series.index)
    for v in vals:
        mask |= (series - v).abs() <= tol
    return mask


df_sel = df[_match(df["frot_Hz"], SELECT_FROT)
            & _match(df["flib_Hz"], SELECT_FLIB)
            & _match(df["fstar"], SELECT_FSTAR)
            & _match(df["dphi_deg"], SELECT_DPHI)].copy()

no_filter = all(s is None for s in
                (SELECT_FROT, SELECT_FLIB, SELECT_FSTAR, SELECT_DPHI))
print("Selected %d of %d runs%s"
      % (len(df_sel), len(df), "  (no filter)" if no_filter else ""))
df_sel[["run", "run idx", "frot_Hz", "flib_Hz", "dphi_deg", "fstar",
        "mean_Ekin", "std_Ekin"]]

Selected 24 of 29 runs


,run,run idx,frot_Hz,flib_Hz,dphi_deg,fstar,mean_Ekin,std_Ekin
0,frot050_flib0400_dphi2deg_SS1,1,0.5,0.400,2.0,0.80,9.420107e-06,2.350569e-06
1,frot050_flib0400_dphi2deg_SS2,2,0.5,0.400,2.0,0.80,7.840482e-06,1.200973e-06
2,frot050_flib0405_dphi2deg_SS1,1,0.5,0.405,2.0,0.81,9.648374e-06,2.192325e-06
3,frot050_flib0410_dphi2deg_SS1,1,0.5,0.410,2.0,0.82,9.535370e-06,1.391610e-06
4,frot050_flib0415_dphi2deg_SS1,1,0.5,0.415,2.0,0.83,1.038518e-05,1.135649e-06
5,frot050_flib0420_dphi2deg_SS1,1,0.5,0.420,2.0,0.84,1.057577e-05,1.100144e-06
6,frot050_flib0425_dphi2deg_SS1,1,0.5,0.425,2.0,0.85,1.212529e-05,2.819907e-06
7,frot050_flib0430_dphi2deg_SS1,1,0.5,0.430,2.0,0.86,1.475152e-05,5.792468e-06
8,frot050_flib0435_dphi2deg_SS1,1,0.5,0.435,2.0,0.87,1.811616e-05,8.590996e-06
9,frot050_flib0440_dphi2deg_SS1,1,0.5,0.440,2.0,0.88,2.497484e-05,1.391982e-05


In [9]:
# Plot the selected runs. Save to OUTPUT_FILE only when no filter is active,
# so a filtered subset never overwrites the full resonance figure.
fig = plot_summary(df_sel, OUTPUT_FILE if no_filter else None)